# 🍅 Tomato Disease CNN — Complete Results Analysis

Full numerical **and** visual evaluation of every trained model.

**How to run:** `Cell → Run All`. The notebook must sit in the project root
(next to `config.py`).

### What's in here

| § | Section | Needs GPU? |
|---|---------|-----------|
| 1 | Setup & environment | no |
| 2 | Dataset overview + sample images | no |
| 3 | Training curves (loss / accuracy per model) | no |
| 4 | **Overfitting analysis** | no |
| 5 | Validation vs test accuracy comparison | no |
| 6 | Efficiency — parameters vs accuracy | no |
| 7 | Test-set inference | **slow on CPU** |
| 8 | Accuracy / precision / recall / F1 per class | after §7 |
| 9 | Confusion matrices | after §7 |
| 10 | Per-class F1 heatmap across all models | after §7 |
| 11 | Misclassified examples | after §7 |
| 12 | Grad-CAM — what the models look at | after §7 |
| 13 | Inference speed | after §7 |
| 14 | Final verdict & recommendations | after §7 |

Sections 1–6 read the saved artefacts in `results/` and are **instant**.
Section 7 runs real inference — see the `MAX_TEST_IMAGES` knob there.

---
## 1. Setup & environment

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader

warnings.filterwarnings("ignore")
%matplotlib inline

# ── Locate the project root (folder containing config.py) ──
HERE = Path.cwd()
PROJECT = HERE if (HERE / "config.py").exists() else HERE.parent
if not (PROJECT / "config.py").exists():
    raise FileNotFoundError(
        f"Could not find config.py from {HERE}. "
        "Put this notebook in the project root, next to config.py."
    )
sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)

import config
from models.model_builder import get_model, count_parameters
from utils.dataset import (TomatoDiseaseDataset, build_samples, split_samples,
                           get_val_transforms, inference_transform)
from utils.gradcam import GradCAM, get_target_layer

# ── Plot styling ──
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titleweight"] = "bold"
PALETTE = sns.color_palette("viridis", 8)

print(f"Project     : {PROJECT}")
print(f"Dataset     : {config.DATA_DIR}")
print(f"Device      : {config.DEVICE}   (AMP: {config.USE_AMP})")
print(f"Torch       : {torch.__version__}")
print(f"Classes     : {config.NUM_CLASSES}")
print(f"Image size  : {config.IMAGE_SIZE}x{config.IMAGE_SIZE}")

In [ ]:
# Which models have a trained checkpoint?
rows = []
for m in config.MODELS_TO_TRAIN:
    best = config.MODELS_DIR / f"{m}_best.pth"
    hist = config.RESULTS_DIR / f"{m}_history.json"
    rows.append({
        "Model": m,
        "Checkpoint": "yes" if best.exists() else "MISSING",
        "Size (MB)": round(best.stat().st_size / 1e6, 1) if best.exists() else 0,
        "History": "yes" if hist.exists() else "MISSING",
    })

inventory = pd.DataFrame(rows)
TRAINED = [r["Model"] for _, r in inventory.iterrows() if r["Checkpoint"] == "yes"]
print(f"{len(TRAINED)} of {len(config.MODELS_TO_TRAIN)} models have checkpoints\n")
inventory

---
## 2. Dataset overview

Class balance matters for reading the metrics later — this dataset is
**imbalanced**, so plain accuracy can be misleading. A model that always
predicts the largest class already scores ~20%.

In [ ]:
samples = build_samples(config.DATA_DIR, verbose=False)
train_s, val_s, test_s = split_samples(samples)

counts = pd.Series([lbl for _, lbl in samples]).value_counts().sort_index()
dist = pd.DataFrame({
    "Class": config.DISPLAY_NAMES,
    "Images": [counts.get(i, 0) for i in range(config.NUM_CLASSES)],
})
dist["Share %"] = (dist["Images"] / dist["Images"].sum() * 100).round(2)
dist = dist.sort_values("Images", ascending=False).reset_index(drop=True)

print(f"Total images : {len(samples):,}")
print(f"Train / Val / Test : {len(train_s):,} / {len(val_s):,} / {len(test_s):,}")
print(f"Largest class share : {dist['Share %'].max():.2f}%  "
      f"<- majority-class baseline accuracy")
dist

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

sns.barplot(data=dist, y="Class", x="Images", ax=axes[0],
            palette="viridis", orient="h")
axes[0].set_title("Images per class (imbalanced)")
for i, v in enumerate(dist["Images"]):
    axes[0].text(v + 30, i, str(v), va="center", fontsize=8)

split_df = pd.DataFrame({"Split": ["Train", "Val", "Test"],
                         "Images": [len(train_s), len(val_s), len(test_s)]})
sns.barplot(data=split_df, x="Split", y="Images", ax=axes[1], palette="crest")
axes[1].set_title("Stratified split (70 / 15 / 15)")
for i, v in enumerate(split_df["Images"]):
    axes[1].text(i, v + 60, f"{v:,}", ha="center", fontsize=9)

plt.tight_layout(); plt.show()

In [ ]:
# One example image per class
fig, axes = plt.subplots(2, 5, figsize=(15, 6.5))
by_class = {}
for p, l in samples:
    by_class.setdefault(l, p)

for i, ax in enumerate(axes.flat):
    ax.imshow(Image.open(by_class[i]).convert("RGB").resize((180, 180)))
    ax.set_title(config.DISPLAY_NAMES[i], fontsize=9)
    ax.axis("off")

plt.suptitle("Sample leaf per class", fontweight="bold")
plt.tight_layout(); plt.show()

---
## 3. Training curves

Loss and accuracy per epoch, straight from `results/<Model>_history.json`.

Two things to keep in mind when reading these:

- **Training loss never reaches 0.** `label_smoothing=0.1` puts a floor around
  ~0.5. A model at 100% accuracy still shows ~0.51 loss. That is by design.
- **Training accuracy can sit *below* validation accuracy.** Training runs with
  augmentation on (rotation, colour jitter, blur, random erasing); validation
  runs clean. A negative gap is healthy, not a bug.

In [ ]:
histories = {}
for m in config.MODELS_TO_TRAIN:
    p = config.RESULTS_DIR / f"{m}_history.json"
    if p.exists():
        histories[m] = json.loads(p.read_text())

print(f"Loaded {len(histories)} training histories: {', '.join(histories)}")

In [ ]:
n = len(histories)
fig, axes = plt.subplots(n, 2, figsize=(13, 3.1 * n))
if n == 1:
    axes = np.array([axes])

for row, (name, h) in enumerate(histories.items()):
    ep = range(1, len(h["train_loss"]) + 1)

    ax = axes[row, 0]
    ax.plot(ep, h["train_loss"], "o-", ms=3, label="train", color="#2b8cbe")
    ax.plot(ep, h["val_loss"],   "s-", ms=3, label="val",   color="#e34a33")
    ax.set_title(f"{name} — Loss"); ax.set_xlabel("epoch"); ax.legend(fontsize=8)

    ax = axes[row, 1]
    ax.plot(ep, [a*100 for a in h["train_acc"]], "o-", ms=3,
            label="train", color="#2b8cbe")
    ax.plot(ep, [a*100 for a in h["val_acc"]], "s-", ms=3,
            label="val", color="#e34a33")
    best = max(h["val_acc"]) * 100
    ax.axhline(best, ls="--", lw=1, color="green", alpha=.7)
    ax.text(1, best + 1, f"best {best:.2f}%", fontsize=8, color="green")
    ax.set_title(f"{name} — Accuracy (%)"); ax.set_xlabel("epoch")
    ax.set_ylim(0, 105); ax.legend(fontsize=8)

plt.tight_layout(); plt.show()

---
## 4. Overfitting analysis

Three independent signals per model:

1. **Generalisation gap** — final train accuracy minus best validation accuracy.
   Large positive = memorising the training set.
2. **Validation-loss trend** — slope over the final third of training.
   Rising while training loss falls is the classic overfitting signature.
3. **Val → test drop** — the honest check, since the test split was never used
   for early stopping or checkpoint selection.

In [ ]:
def analyse(name, h):
    tr, va = h["train_acc"], h["val_acc"]
    tl, vl = h["train_loss"], h["val_loss"]
    n_ep = len(va)
    gap = tr[-1] - max(va)

    k = max(3, n_ep // 3)
    late = vl[-k:]
    slope = float(np.polyfit(range(len(late)), late, 1)[0]) if len(late) > 1 else 0.0

    if max(va) < 0.30:
        verdict = "FAILED TO TRAIN"
    elif slope > 0.002 and gap > 0.03:
        verdict = "Overfitting"
    elif slope > 0.002:
        verdict = "Mild — val loss rising"
    elif gap > 0.05:
        verdict = "Slight — train >> val"
    else:
        verdict = "Healthy"

    return {
        "Model": name,
        "Epochs": n_ep,
        "Final train acc %": round(tr[-1] * 100, 2),
        "Best val acc %": round(max(va) * 100, 2),
        "Gap (train-val) %": round(gap * 100, 2),
        "Val-loss slope": round(slope, 5),
        "Verdict": verdict,
    }

overfit = pd.DataFrame([analyse(n, h) for n, h in histories.items()])
overfit = overfit.sort_values("Best val acc %", ascending=False).reset_index(drop=True)


def colour(v):
    return {
        "Healthy":              "background-color:#d4f4d4",
        "Slight — train >> val":"background-color:#fff4cc",
        "Mild — val loss rising":"background-color:#ffe0b3",
        "Overfitting":          "background-color:#ffcccc",
        "FAILED TO TRAIN":      "background-color:#ff9999;font-weight:bold",
    }.get(v, "")


# Styler.applymap was removed in pandas 3.0 and replaced by Styler.map.
# Pick whichever this pandas has, so the notebook runs on old and new alike.
_sty = overfit.style
_cellwise = getattr(_sty, "map", None) or _sty.applymap
_cellwise(colour, subset=["Verdict"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

o = overfit.sort_values("Gap (train-val) %")
cols = ["#e34a33" if g > 5 else "#31a354" for g in o["Gap (train-val) %"]]
axes[0].barh(o["Model"], o["Gap (train-val) %"], color=cols)
axes[0].axvline(0, color="black", lw=1)
axes[0].axvline(5, color="red", ls="--", lw=1, label="overfit threshold (+5%)")
axes[0].set_title("Generalisation gap\n(train acc − best val acc)")
axes[0].set_xlabel("percentage points"); axes[0].legend(fontsize=8)

for name, h in histories.items():
    if max(h["val_acc"]) < 0.3:
        continue
    ep = range(1, len(h["val_loss"]) + 1)
    axes[1].plot(ep, h["val_loss"], label=name, lw=1.6)
axes[1].set_title("Validation loss — rising tail = overfitting")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("val loss")
axes[1].legend(fontsize=7)

plt.tight_layout(); plt.show()

---
## 5. Validation vs test accuracy

`model_comparison.json` is written by `train.py` (validation);
`test_results.json` by `evaluate.py --model all` (held-out test set).

In [ ]:
def load_json(p):
    p = Path(p)
    if not p.exists():
        return {}
    try:
        return json.loads(p.read_text())
    except json.JSONDecodeError:
        return {}

val_res  = load_json(config.RESULTS_DIR / "model_comparison.json")
test_res = load_json(config.RESULTS_DIR / "test_results.json")

if not test_res:
    print("No test_results.json — run:  python evaluate.py --model all")

comp = pd.DataFrame({
    "Model": list(config.MODELS_TO_TRAIN),
    "Val acc %":  [round(val_res.get(m, np.nan) * 100, 2) if m in val_res else np.nan
                   for m in config.MODELS_TO_TRAIN],
    "Test acc %": [round(test_res.get(m, np.nan) * 100, 2) if m in test_res else np.nan
                   for m in config.MODELS_TO_TRAIN],
})
comp["Val→Test drop"] = (comp["Val acc %"] - comp["Test acc %"]).round(2)
comp = comp.sort_values("Test acc %", ascending=False).reset_index(drop=True)
comp

In [ ]:
plot_df = comp.dropna(subset=["Test acc %"])
if len(plot_df):
    x = np.arange(len(plot_df)); w = 0.38
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(x - w/2, plot_df["Val acc %"],  w, label="Validation", color="#4c9f70")
    ax.bar(x + w/2, plot_df["Test acc %"], w, label="Test",       color="#2b6cb0")

    for i, (v, t) in enumerate(zip(plot_df["Val acc %"], plot_df["Test acc %"])):
        if not np.isnan(v): ax.text(i - w/2, v + 1, f"{v:.1f}", ha="center", fontsize=8)
        if not np.isnan(t): ax.text(i + w/2, t + 1, f"{t:.1f}", ha="center", fontsize=8)

    baseline = dist["Share %"].max()
    ax.axhline(baseline, ls="--", color="red", lw=1.2,
               label=f"majority-class baseline ({baseline:.1f}%)")
    ax.set_xticks(x); ax.set_xticklabels(plot_df["Model"], rotation=20, ha="right")
    ax.set_ylabel("Accuracy (%)"); ax.set_ylim(0, 108)
    ax.set_title("Model comparison — validation vs test")
    ax.legend()
    plt.tight_layout(); plt.show()

---
## 6. Efficiency — accuracy vs model size

Bigger is not better. This is the plot that tells you which model to actually
deploy: top-left is the sweet spot (high accuracy, few parameters).

In [ ]:
params = {}
for m in config.MODELS_TO_TRAIN:
    mdl = get_model(m, config.NUM_CLASSES, pretrained=False)
    params[m] = sum(p.numel() for p in mdl.parameters())
    del mdl

eff = comp.copy()
eff["Params (M)"] = [round(params[m] / 1e6, 1) for m in eff["Model"]]
eff = eff.dropna(subset=["Test acc %"])

fig, ax = plt.subplots(figsize=(10, 6))
for i, r in eff.iterrows():
    ax.scatter(r["Params (M)"], r["Test acc %"], s=190,
               color="#e34a33" if r["Test acc %"] < 50 else "#31a354",
               edgecolor="black", zorder=3)
    ax.annotate(r["Model"], (r["Params (M)"], r["Test acc %"]),
                xytext=(7, 5), textcoords="offset points", fontsize=9)

ax.set_xscale("log")
ax.set_xlabel("Parameters (millions, log scale)")
ax.set_ylabel("Test accuracy (%)")
ax.set_title("Accuracy vs model size — top-left is best")
plt.tight_layout(); plt.show()

eff[["Model", "Params (M)", "Test acc %"]].reset_index(drop=True)

---
## 7. Test-set inference

Everything below (confusion matrices, precision/recall, misclassified examples)
is computed here.

> ### ⚙️ Set this before running
> `MAX_TEST_IMAGES` controls how many test images to evaluate.
> - **On CPU** keep it at 500 — the full set takes hours.
> - **On GPU** set it to `None` to use all 2,411 test images.

In [ ]:
MAX_TEST_IMAGES = 500      # None = full test split
BATCH = 32                 # lower this if you hit out-of-memory

# Stratified subsample so every class stays represented
if MAX_TEST_IMAGES is None or MAX_TEST_IMAGES >= len(test_s):
    subset = list(test_s)
else:
    per = max(1, MAX_TEST_IMAGES // config.NUM_CLASSES)
    buckets = {}
    for p, l in test_s:
        buckets.setdefault(l, []).append((p, l))
    subset = [it for l in sorted(buckets) for it in buckets[l][:per]]

eval_ds = TomatoDiseaseDataset(subset, get_val_transforms())
# num_workers=0 — worker processes are unreliable inside notebooks on Windows
eval_dl = DataLoader(eval_ds, batch_size=BATCH, shuffle=False, num_workers=0)

y_true = np.array([l for _, l in subset])
paths  = [p for p, _ in subset]
print(f"Evaluating on {len(subset)} of {len(test_s)} test images "
      f"({len(subset)/len(test_s)*100:.0f}%)")

In [ ]:
PRED = {}   # model -> {"pred":…, "prob":…, "acc":…, "ms_per_img":…}

for name in TRAINED:
    ck = torch.load(config.MODELS_DIR / f"{name}_best.pth",
                    map_location=config.DEVICE, weights_only=False)
    model = get_model(name, config.NUM_CLASSES, pretrained=False)
    model.load_state_dict(ck["model_state"])
    model.to(config.DEVICE).eval()

    probs = []
    t0 = time.time()
    with torch.no_grad():
        for xb, _ in eval_dl:
            out = model(xb.to(config.DEVICE))
            probs.append(F.softmax(out, dim=1).cpu())
    elapsed = time.time() - t0

    prob = torch.cat(probs).numpy()
    pred = prob.argmax(1)
    PRED[name] = {
        "pred": pred,
        "prob": prob,
        "acc": float((pred == y_true).mean()),
        "ms_per_img": elapsed / len(subset) * 1000,
        "n_classes_predicted": len(set(pred.tolist())),
    }
    print(f"  {name:<16} acc {PRED[name]['acc']*100:6.2f}%   "
          f"{PRED[name]['ms_per_img']:6.1f} ms/img   "
          f"predicts {PRED[name]['n_classes_predicted']}/10 classes")

    del model, ck
    if config.DEVICE.type == "cuda":
        torch.cuda.empty_cache()

print("\nDone.")

---
## 8. Accuracy, precision, recall, F1

**Macro average** treats every class equally — this is the number to trust on an
imbalanced dataset. **Weighted average** is dominated by the big classes and will
flatter a model that only learned those.

In [ ]:
from sklearn.metrics import (classification_report, accuracy_score,
                             precision_recall_fscore_support, confusion_matrix)

summary = []
for name, d in PRED.items():
    p_m, r_m, f_m, _ = precision_recall_fscore_support(
        y_true, d["pred"], average="macro", zero_division=0)
    p_w, r_w, f_w, _ = precision_recall_fscore_support(
        y_true, d["pred"], average="weighted", zero_division=0)
    summary.append({
        "Model": name,
        "Accuracy %": round(d["acc"] * 100, 2),
        "Precision (macro) %": round(p_m * 100, 2),
        "Recall (macro) %":    round(r_m * 100, 2),
        "F1 (macro) %":        round(f_m * 100, 2),
        "F1 (weighted) %":     round(f_w * 100, 2),
    })

metrics = (pd.DataFrame(summary)
             .sort_values("F1 (macro) %", ascending=False)
             .reset_index(drop=True))
metrics.style.background_gradient(cmap="RdYlGn", vmin=0, vmax=100,
                                  subset=metrics.columns[1:])

In [ ]:
# Per-class breakdown for every model
for name in metrics["Model"]:
    rep = classification_report(
        y_true, PRED[name]["pred"],
        labels=list(range(config.NUM_CLASSES)),
        target_names=config.DISPLAY_NAMES,
        output_dict=True, zero_division=0)
    df = pd.DataFrame(rep).T[["precision", "recall", "f1-score", "support"]]
    df[["precision", "recall", "f1-score"]] *= 100
    print(f"\n{'='*70}\n  {name}   —   accuracy {PRED[name]['acc']*100:.2f}%\n{'='*70}")
    display(df.round(2).style.background_gradient(
        cmap="RdYlGn", vmin=0, vmax=100,
        subset=["precision", "recall", "f1-score"]))

---
## 9. Confusion matrices

Row-normalised: each row sums to 100%, so the diagonal is per-class recall.
Bright off-diagonal cells are the classes a model confuses.

In [ ]:
names = list(PRED)
ncol = 2
nrow = int(np.ceil(len(names) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(15, 6.2 * nrow))
axes = np.atleast_1d(axes).flatten()

short = [n[:14] for n in config.DISPLAY_NAMES]
for ax, name in zip(axes, names):
    cm = confusion_matrix(y_true, PRED[name]["pred"],
                          labels=list(range(config.NUM_CLASSES)))
    cmn = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1) * 100
    sns.heatmap(cmn, annot=True, fmt=".0f", cmap="viridis", vmin=0, vmax=100,
                xticklabels=short, yticklabels=short, ax=ax,
                cbar_kws={"label": "% of true class"}, annot_kws={"size": 7})
    ax.set_title(f"{name} — {PRED[name]['acc']*100:.2f}%")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.tick_params(labelsize=7)

for ax in axes[len(names):]:
    ax.axis("off")

plt.tight_layout(); plt.show()

---
## 10. Per-class F1 across all models

Which classes are hard for *every* model? Dark columns are the genuinely
difficult diseases; dark rows are broken models.

In [ ]:
f1_rows = {}
for name in PRED:
    _, _, f1, _ = precision_recall_fscore_support(
        y_true, PRED[name]["pred"],
        labels=list(range(config.NUM_CLASSES)), zero_division=0)
    f1_rows[name] = f1 * 100

f1_df = pd.DataFrame(f1_rows, index=config.DISPLAY_NAMES).T
f1_df = f1_df.loc[f1_df.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(13, 0.55 * len(f1_df) + 3))
sns.heatmap(f1_df, annot=True, fmt=".1f", cmap="RdYlGn", vmin=0, vmax=100,
            linewidths=.5, ax=ax, cbar_kws={"label": "F1 (%)"},
            annot_kws={"size": 8})
ax.set_title("Per-class F1 score — models (rows) vs classes (columns)")
plt.xticks(rotation=35, ha="right"); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

print("Hardest classes (mean F1 across all working models):")
working = [n for n in PRED if PRED[n]["n_classes_predicted"] > 1]
if working:
    display(f1_df.loc[working].mean().sort_values().head(5).round(2).to_frame("Mean F1 %"))

---
## 11. Misclassified examples

The best model's actual mistakes, sorted by how confidently it got them wrong.
Confident errors are the ones worth looking at — they usually reveal genuinely
ambiguous images or mislabelled data.

In [ ]:
working = [n for n in PRED if PRED[n]["n_classes_predicted"] > 1]
BEST = max(working, key=lambda n: PRED[n]["acc"]) if working else None
print(f"Best model: {BEST}")

if BEST:
    pred = PRED[BEST]["pred"]; prob = PRED[BEST]["prob"]
    wrong = np.where(pred != y_true)[0]
    print(f"{len(wrong)} misclassified out of {len(y_true)} "
          f"({len(wrong)/len(y_true)*100:.2f}%)")

    if len(wrong):
        conf = prob[wrong, pred[wrong]]
        order = wrong[np.argsort(-conf)][:8]
        cols = min(4, len(order))
        rows = int(np.ceil(len(order) / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(3.6 * cols, 4.0 * rows))
        for ax, idx in zip(np.atleast_1d(axes).flatten(), order):
            ax.imshow(Image.open(paths[idx]).convert("RGB").resize((190, 190)))
            ax.set_title(
                f"true: {config.DISPLAY_NAMES[y_true[idx]]}\n"
                f"pred: {config.DISPLAY_NAMES[pred[idx]]} "
                f"({prob[idx, pred[idx]]*100:.0f}%)",
                fontsize=8, color="darkred")
            ax.axis("off")
        for ax in np.atleast_1d(axes).flatten()[len(order):]:
            ax.axis("off")
        plt.suptitle(f"{BEST} — most confident mistakes", fontweight="bold")
        plt.tight_layout(); plt.show()
    else:
        print("No misclassifications in this subset.")

---
## 12. Grad-CAM — what is the model actually looking at?

Red = regions that drove the prediction. A healthy model highlights **lesions on
the leaf**. A model highlighting background or leaf edges has learned a shortcut
and will not survive real-world photos.

In [ ]:
N_IMAGES = 3
sample_idx = [int(i) for i in np.linspace(0, len(subset) - 1, N_IMAGES).astype(int)]
show_models = [m for m in (working or list(PRED))][:4]

if show_models:
    fig, axes = plt.subplots(len(sample_idx), len(show_models) + 1,
                             figsize=(3.1 * (len(show_models) + 1), 3.3 * len(sample_idx)))
    axes = np.atleast_2d(axes)

    for r, idx in enumerate(sample_idx):
        img = Image.open(paths[idx]).convert("RGB")
        axes[r, 0].imshow(img.resize((190, 190)))
        axes[r, 0].set_title(f"TRUE:\n{config.DISPLAY_NAMES[y_true[idx]]}",
                             fontsize=9, fontweight="bold")
        axes[r, 0].axis("off")

        tensor = inference_transform(img).unsqueeze(0)
        for c, name in enumerate(show_models, start=1):
            ck = torch.load(config.MODELS_DIR / f"{name}_best.pth",
                            map_location=config.DEVICE, weights_only=False)
            model = get_model(name, config.NUM_CLASSES, pretrained=False)
            model.load_state_dict(ck["model_state"])
            model.to(config.DEVICE).eval()

            cam = GradCAM(model, get_target_layer(model, name))
            _, overlay, cls, conf = cam(tensor.clone())
            axes[r, c].imshow(overlay[:, :, ::-1])   # BGR -> RGB
            ok = (cls == y_true[idx])
            axes[r, c].set_title(
                f"{name}\n{config.DISPLAY_NAMES[cls]} ({conf*100:.0f}%)",
                fontsize=8, color="green" if ok else "darkred")
            axes[r, c].axis("off")

            del model, ck
            if config.DEVICE.type == "cuda":
                torch.cuda.empty_cache()

    plt.suptitle("Grad-CAM — red = evidence the model used", fontweight="bold")
    plt.tight_layout(); plt.show()

---
## 13. Inference speed

Matters for `camera_app.py` — real-time needs roughly < 33 ms/image for 30 fps.

In [ ]:
speed = pd.DataFrame([
    {"Model": n,
     "ms / image": round(d["ms_per_img"], 2),
     "images / sec": round(1000 / d["ms_per_img"], 1),
     "Test acc %": round(d["acc"] * 100, 2)}
    for n, d in PRED.items()
]).sort_values("ms / image").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 4.5))
cols = ["#31a354" if m < 33 else "#fdae61" if m < 100 else "#e34a33"
        for m in speed["ms / image"]]
ax.barh(speed["Model"], speed["ms / image"], color=cols)
ax.axvline(33, ls="--", color="red", lw=1.2, label="30 fps budget (33 ms)")
ax.set_xlabel(f"ms per image  (device: {config.DEVICE})")
ax.set_title("Inference speed — lower is better"); ax.legend()
plt.tight_layout(); plt.show()

speed

---
## 14. Final verdict

In [ ]:
print("=" * 74)
print("  FINAL REPORT")
print("=" * 74)

broken = [n for n, d in PRED.items() if d["n_classes_predicted"] <= 1]
ok     = [n for n, d in PRED.items() if d["n_classes_predicted"] > 1]

print(f"\n  Evaluated on {len(subset)} test images "
      f"({'FULL test split' if len(subset) == len(test_s) else 'subset'})")
print(f"  Working models : {len(ok)} / {len(PRED)}")

if ok:
    best = max(ok, key=lambda n: PRED[n]["acc"])
    fastest = min(ok, key=lambda n: PRED[n]["ms_per_img"])
    print(f"\n  Most accurate  : {best}  ({PRED[best]['acc']*100:.2f}%)")
    print(f"  Fastest        : {fastest}  ({PRED[fastest]['ms_per_img']:.1f} ms/img)")
    print(f"\n  -> config.py:  INFERENCE_MODEL = \"{best}\"")
    print(f"  -> camera use: INFERENCE_MODEL = \"{fastest}\"")

if broken:
    print(f"\n  {'!'*66}")
    print(f"  BROKEN — predict a single class for every input:")
    for b in broken:
        maj = config.DISPLAY_NAMES[int(PRED[b]['pred'][0])]
        print(f"      {b:<14} always predicts '{maj}'  "
              f"({PRED[b]['acc']*100:.2f}%)")
    print(f"\n  These collapsed to the majority class. VGG16 and AlexNet have no")
    print(f"  BatchNorm, so AdamW at lr=1e-3 diverges. Retrain them lower:")
    for b in broken:
        print(f"      python train.py --model {b} --lr 1e-4")
    print(f"  {'!'*66}")

if not overfit.empty:
    bad = overfit[overfit["Verdict"].isin(["Overfitting", "Mild — val loss rising"])]
    print(f"\n  Overfitting    : "
          f"{'none detected' if bad.empty else ', '.join(bad['Model'])}")

print("\n  NOTE: PlantVillage images are lab-captured on uniform backgrounds.")
print("  Accuracy here is an upper bound — expect a real drop on field photos.")
print("=" * 74)

---
### Saving results

Every table above is a pandas DataFrame — export any of them with:

```python
metrics.to_csv("results/notebook_metrics.csv", index=False)
overfit.to_csv("results/notebook_overfitting.csv", index=False)
```

Save any figure by adding `plt.savefig("results/my_plot.png", dpi=200,
bbox_inches="tight")` before its `plt.show()`.